In [ ]:
%load_ext autoreload
%autoreload 3

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import mkdir
import json
from PIL import Image
from pycocotools import mask as mask_utils
from tqdm import tqdm
from mtrain.disk import DiskBooleanMask, DiskImage
import itertools

In [ ]:
DS = Path("../../datasets/")
BASE = DS / "test-samples"
NEG_MASKING_V1 = BASE / "neg-masking" / "V1"
TRASH = NEG_MASKING_V1 / "trash"
SAMPLES_MAPILLARY = NEG_MASKING_V1 / "samples_mapillary"

CLIP_FILE_NAMES = ["clip_bottles.txt", "clip_litter.txt", "clip_plastic.txt", "clip_tobacco_packs.txt", "clip_delhi_litter.txt"]
CLIP_FILES = [TRASH / c for c in CLIP_FILE_NAMES]
print("all clip files exist:", all([f.exists() for f in CLIP_FILES]))

SAMPLES_MAPILLARY.exists(), TRASH.exists()

In [ ]:
from mtrain.utils import show

def read_clip_file(path):
    with open(path) as f:
        lines = f.readlines()
    imgs = [Path(line.split("\t")[1].strip()) for line in lines]
    return imgs

def view_clip_file_content(path, n=12):
    print(path.name)
    imgs = read_clip_file(path)
    imgs = [plt.imread(i) for i in imgs[:n]]
    show(imgs, ncols=4)

# clip check

## Samples litter file

In [ ]:
view_clip_file_content(CLIP_FILES[1])

In [ ]:
from fastai.vision.all import load_learner
from mtrain.smallnet.unet.predict.strided import single


learner100 = load_learner(
    "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/enguled-bbox-levels-crops-v3/log/export_iter_14.pkl"
)

learner50 = load_learner(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/iter_4_engulf_t009_more-skew-resnet18-50x50-v2/model.pkl"
)

In [ ]:
from mtrain.disk import DiskImage

images = read_clip_file(CLIP_FILES[1])

def run_model_on_images(images, learner, size, strides, bs=4):
    res = []
    for i in tqdm(images):
        img = DiskImage.load(i)
        r = single.strided_predict_unet_only_mask(img, size, learner, strides, bs)
        res.append((img, r))
    return res

In [ ]:
res_100 = run_model_on_images(images[:10], learner100, 100, [50])
res_50 = run_model_on_images(images[:10], learner50, 50, [25])

In [ ]:
res_100.extend(run_model_on_images([images[12]], learner100, 100, [50]))

In [ ]:
res_50.extend(run_model_on_images([images[12]], learner50, 50, [25]))

In [ ]:
from mtrain.utils import show
from mtrain.smallnet.unet.predict import overlay_mask_on_img

O = overlay_mask_on_img

def show_both_reses(res1, res2):
    res = []
    for r1, r2 in zip(res1, res2):
        img = r1[0]
        m1 = r1[1]
        m2 = r2[1]
        res.append(
            (img, O(img, m1), O(img, m2))
        )
    to_show = list(itertools.chain.from_iterable(res))
    show(to_show, ncols=3)

        

def show_reses(res):
    to_show = [(img, overlay_mask_on_img(img, mask)) for (img, mask) in res]
    to_show = list(itertools.chain.from_iterable(to_show))
    show(to_show)

In [ ]:
def _get_idxes(r, idxes):
    return [r[i] for i in idxes]
idxes = [2,4,6,8,10]
show_both_reses(_get_idxes(res_100, idxes), _get_idxes(res_50, idxes))

## Delhi litter file

In [ ]:
idxes = [
    0, 1, 6, 7, 9, 14, 16, 21
]

In [ ]:
image_paths = read_clip_file(CLIP_FILES[-1])
image_paths = _get_idxes(image_paths, idxes)

In [ ]:
res100 = run_model_on_images(image_paths, learner100, 100, [50])
res50 = run_model_on_images(image_paths, learner50, 50, [25])

In [ ]:
show_both_reses(res100, res50)

In [ ]:
view_clip_file_content(CLIP_FILES[-1], 24)

# Prepare dataset for delhi

this is extra data that i would use, apart frmo the already exisitng data

In [ ]:
images = read_clip_file(CLIP_FILES[-1])

In [ ]:
OUT = mkdir(TRASH / "delhi_litter")

def save_delhi_litter_masks(out, images, learner, size, strides):
    for im in tqdm(images):
        img = DiskImage.load(im)
        mask = single.strided_predict_unet_only_mask(img, size, learner, strides, 2)
        dest = mkdir(out / im.stem)
        DiskImage.save(img, dest / "image.jpg")
        DiskBooleanMask.save(mask, dest / "mask.png")
    

In [ ]:
save_delhi_litter_masks(OUT, images[50:100], learner100, 100, [50])

In [ ]:
from mtrain.seg import mapillary as mapi, elevated_vegetation as elev

MAPI_LABELS_TO_EXCLUDE = [
    mapi.Label.PERSON,
    mapi.Label.MOTORCYCLIST,
    mapi.Label.BICYCLIST,
    mapi.Label.GROUND_ANIMAL,
    mapi.Label.OTHER_RIDER,
    mapi.Label.BIRD,

    mapi.Label.SKY,

    mapi.Label.BOAT,
    mapi.Label.BUS,
    mapi.Label.CAR,
    mapi.Label.CARAVAN,
    mapi.Label.MOTORCYCLE,
    mapi.Label.ON_RAILS,
    mapi.Label.OTHER_VEHICLE,
    mapi.Label.EGO_VEHICLE,
    mapi.Label.TRAILER,
    mapi.Label.TRUCK,
    mapi.Label.WHEELED_SLOW,
    mapi.Label.CAR_MOUNT,
    mapi.Label.BICYCLE,
    mapi.Label.BRIDGE,
    mapi.Label.TUNNEL,

    mapi.Label.BUILDING,
    mapi.Label.BILLBOARD,
    mapi.Label.BANNER,
    mapi.Label.STREET_LIGHT,
    mapi.Label.JUNCTION_BOX,
    mapi.Label.MAILBOX,
    mapi.Label.MOUNTAIN,
    mapi.Label.PHONE_BOOTH,
    mapi.Label.TRAFFIC_SIGN_FRONT,
    mapi.Label.TRAFFIC_SIGN_FRAME,
    mapi.Label.TRAFFIC_SIGN_BACK,

]

ELEV_LABELS_TO_EXCLUDE = [elev.Label.ELEVATED_VEGETATION]

def get_trimmed_mask(mask, elev_pred, mapi_pred):
    # for now, we remove all obvious things that we see
    # then we will go through all the remaining masks 
    # in decreasing order of the amount of segmentation
    # and remove more stuff
    if mapi_pred is not None:
        mapi_exclude_mask = mapi.get_mask_with_labels(mapi_pred, MAPI_LABELS_TO_EXCLUDE)
    else:
        mapi_exclude_mask = np.zeros(mask.shape, dtype=bool)
    if elev_pred is not None:
        elev_exclude_mask = elev.get_mask_with_labels(elev_pred, ELEV_LABELS_TO_EXCLUDE)
    else:
        elev_exclude_mask = np.zeros(mask.shape, dtype=bool)

    return mask & (~mapi_exclude_mask) & (~elev_exclude_mask)

def get_panoptic_masks(image_path: Path):
    mapi_pred = mapi.cached_predict(image_path)
    elev_pred = elev.cached_predict(image_path)
    return mapi_pred, elev_pred

def get_artifacts(image_dir: Path, orig_mask_name: str = "mask.png"):
    img = DiskImage.load(image_dir / "image.jpg")
    mask = DiskBooleanMask.load(image_dir / orig_mask_name)
    elev_pred, mapi_pred = get_panoptic_masks(image_dir / "image.jpg")
    out = get_trimmed_mask(mask, elev_pred, mapi_pred)

    return {
        "mask": mask,
        "img": img,
        "mapi_pred": mapi_pred,
        "elev_pred": elev_pred,
        "out_mask": out,
    }

In [ ]:
dirs = (d for d in OUT.glob("*") if d.is_dir() and (d / "image.jpg").exists())
# sort these in the order shown in VSCode explorer
# for easy testing
dirs = sorted(dirs, key=lambda p: int(p.name))

In [ ]:
import cv2
import numpy as np

def extract_regions(mask: np.ndarray) -> list[dict]:
    """
    Extract connected components from a binary mask.
    Returns list of dicts with: label, area, bbox, centroid, contour
    """
    mask_u8 = (mask > 0).astype(np.uint8)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)

    regions = []
    for i in range(1, num_labels):  # skip background (0)
        x, y, w, h, area = stats[i]
        cx, cy = centroids[i]
        component_mask = (labels == i).astype(np.uint8)
        contours, _ = cv2.findContours(component_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        regions.append({
            "label": i,
            "area": int(area),
            "bbox": (int(x), int(y), int(w), int(h)),
            "centroid": (float(cx), float(cy)),
            "contour": contours[0] if contours else None,
            "component_mask": component_mask,
        })
    return regions


def region_stats(regions: list[dict]) -> dict:
    """
    Print and return stats about extracted regions.
    """
    areas = np.array([r["area"] for r in regions])
    
    percentiles = [10, 25, 50, 75, 90, 95, 99]
    pct_values = np.percentile(areas, percentiles)

    stats = {
        "num_regions": len(regions),
        "total_area": int(areas.sum()),
        "mean_area": float(areas.mean()),
        "median_area": float(np.median(areas)),
        "std_area": float(areas.std()),
        "min_area": int(areas.min()),
        "max_area": int(areas.max()),
        "percentiles": {p: float(v) for p, v in zip(percentiles, pct_values)},
    }

    print(f"Regions      : {stats['num_regions']}")
    print(f"Total area   : {stats['total_area']}")
    print(f"Mean / Std   : {stats['mean_area']:.1f} / {stats['std_area']:.1f}")
    print(f"Min / Max    : {stats['min_area']} / {stats['max_area']}")
    print("Percentiles  :")
    for p, v in stats["percentiles"].items():
        print(f"  p{p:>2}: {v:.1f}")
    return stats


def print_mask_stats(mask):
    regions = extract_regions(mask)
    region_stats(regions)

def get_mask_with_area_greater(mask, threshold):
    mask = mask.astype(bool)
    regions = extract_regions(mask)
    regions = [r for r in regions if r["area"] >= threshold]
    res = np.zeros(mask.shape, bool)
    for r in regions:
        res |= r["component_mask"].astype(bool)
    return res

In [ ]:
AREA_THRES = 10
import json

def filter_and_save_masks(meta_dir, dirs, label: str, mask_dest_name: str):
    area_and_img_id = []
    for d in tqdm(dirs):
        art = get_artifacts(d)
        res = get_mask_with_area_greater(art["out_mask"], AREA_THRES)
        area = res.sum()
        area_and_img_id.append((area, d.name))

        DiskBooleanMask.save(res, d / f"{mask_dest_name}.png")
    
    stats_f = mkdir(meta_dir) / f"{label}.json"
    stats = {
        "areas": [(int(a),i) for (a,i) in area_and_img_id],
    }
    with open(stats_f, "w") as f:
        json.dump(stats, f)
    


In [ ]:
META = TRASH / "meta"
filter_and_save_masks(META, dirs, "m2", "m2")